# 04 — Extraction Surfaces

**Purpose:** Show every way to pull activations out of a model without keeping a full `Trace`:
`tl.extract` (single forward, named layers -> tensor dict), `tl.batched_extract` (multi-sample),
and `Container` (structured output reconstruction when the model returns dicts/tuples).  Compare
these with `tl.trace` to make the trade-off explicit.

**Surfaces covered:**
- [ ] `tl.extract(model, x, layers)` — basic, list input
- [ ] `tl.extract(model, x, {alias: label})` — dict-mapping / rename
- [ ] `tl.batched_extract(model, stimuli, layers, batch_size)` — multi-sample batching
- [ ] `tl.Container` — structured output view (requires `capture_container_structure=True`)
- [ ] `Container.kind`, `.path`, `.reconstructable`, `.summary()`
- [ ] `Container.__getitem__` — navigate dict/tuple leaves
- [ ] `Container.reconstruct()` — rebuild the original Python container
- [ ] `trace.reconstruct_container()` — top-level convenience
- [ ] `tl.register_container` — custom container registration
- [ ] `Trace.to_pandas()` and `Trace.output_table()` — tabular views
- [ ] `Layer.to_pandas()` and `Op.to_pandas()` — per-record tabular views
- [ ] How `extract` differs from a full `trace`

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
import torchlens.data_classes.container as _cc
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. `tl.extract` — single forward, named layers -> tensor dict

`extract` runs ONE forward pass and returns a plain `dict[str, Tensor]`.  No `Trace` object,
no per-op metadata — just the raw activations.  Fastest path when you only need values.

In [ ]:
model, x = ZOO["tiny_mlp"]()

# First do a trace to discover valid layer labels
trace = tl.trace(model, x)
print("layer_labels:", trace.layer_labels)
print()

In [ ]:
# Extract a single layer
result_single = tl.extract(model, x, ["relu_1_2"])
print("extract([relu_1_2]) return type:", type(result_single).__name__)
print("keys:", list(result_single.keys()))
print("shape:", result_single["relu_1_2"].shape)
print("value:", result_single["relu_1_2"])

In [ ]:
# Extract multiple layers
result_multi = tl.extract(model, x, ["relu_1_2", "linear_2_3"])
print("extract([relu_1_2, linear_2_3]):")
for name, tensor in result_multi.items():
    print(f"  {name}: shape={tensor.shape}  dtype={tensor.dtype}")

## 3. `tl.extract` with dict mapping — alias output keys

Pass `{alias: real_label}` to rename output keys in one step.

In [ ]:
# Dict mapping: key = desired alias, value = real TorchLens layer label
result_renamed = tl.extract(
    model,
    x,
    {"post_relu": "relu_1_2", "pre_output": "linear_2_3"},
)
print("extract with alias dict:")
for k, v in result_renamed.items():
    print(f"  '{k}': shape={v.shape}")

## 4. `tl.batched_extract` — multi-sample extraction

`batched_extract` accepts a dataset-style tensor `[N, *feature]`, chunks it into
`batch_size` slices, runs extract on each, and concatenates results.

In [ ]:
# 10 samples, feature dim 8
stimuli = torch.randn(10, 8)

batched_result = tl.batched_extract(
    model,
    stimuli,
    ["relu_1_2", "linear_2_3"],
    batch_size=4,
    progress=False,  # suppress tqdm for clean notebook output
)
print("batched_extract result type:", type(batched_result).__name__)
print("keys:", list(batched_result.keys()))
for name, tensor in batched_result.items():
    print(f"  {name}: shape={tensor.shape}  (all 10 samples concatenated)")

## 5. `extract` vs `trace` — the trade-off

| | `tl.extract` | `tl.trace` |
|---|---|---|
| Return type | `dict[str, Tensor]` | `Trace` |
| Overhead | Minimal | Full metadata |
| Metadata (shapes, dtypes, graph) | No | Yes |
| Per-op timing, FLOPs, RNG | No | Yes |
| Intervention / replay | No | Yes |
| Save/load (.tlspec) | No | Yes |
| When to use | Repeated activation extraction | One-time inspection / debugging |


In [ ]:
# Show the difference concretely
extract_result = tl.extract(model, x, ["relu_1_2"])
trace_result = tl.trace(model, x)

print("extract type:", type(extract_result).__name__)  # dict
print("trace  type:", type(trace_result).__name__)  # Trace
print()
print("extract['relu_1_2'] type:", type(extract_result["relu_1_2"]).__name__)  # Tensor
print("trace['relu_1_2']   type:", type(trace_result["relu_1_2"]).__name__)  # Op
print()
print("Only trace has:")
print("  .layers:  ", type(trace_result.layers).__name__)
print("  .modules: ", type(trace_result.modules).__name__)
print("  .params:  ", type(trace_result.params).__name__)
print("  .summary():", type(trace_result.summary()).__name__)

## 6. `Container` — structured output reconstruction

When a model returns a `dict` or `tuple`, TorchLens can capture the structure and let you
navigate it via `Container`.  Requires `capture_container_structure=True` on the trace call.

In [ ]:
# Use a model that returns a dict output
dict_model, dict_x = ZOO["dict_output"]()

# Must pass capture_container_structure=True to populate Container objects
dict_trace = tl.trace(dict_model, dict_x, capture_container_structure=True)
print("layer_labels:", dict_trace.layer_labels)
print()

In [ ]:
# Get a Container from the output ops
out_op = dict_trace["output_1"]
container = _cc.container_from_op(out_op)

print("container type:", type(container).__name__)
print("container.kind:", container.kind)
print("len(container):", len(container))
print("container.reconstructable:", container.reconstructable)
print("container.supports_reconstruct:", container.supports_reconstruct)
print()
print("Container.summary():")
print(container.summary())

In [ ]:
# Navigate keys
print("container['a'] (an Op):")
leaf_a = container["a"]
print(repr(leaf_a))
print()
print("leaf shape:", leaf_a.out.shape)
print("leaf value:", leaf_a.out)

In [ ]:
# Reconstruct the original Python dict from captured tensors
reconstructed = container.reconstruct()
print("reconstructed type:", type(reconstructed).__name__)
print("reconstructed:", reconstructed)

In [ ]:
# Convenience: trace.reconstruct_container() with role selector
from torchlens.data_classes.container import Role

reconstructed2 = dict_trace.reconstruct_container(role=Role.MODEL_OUTPUT)
print("reconstruct_container(role=MODEL_OUTPUT):", reconstructed2)

In [ ]:
# Tuple output model
tuple_model, tuple_x = ZOO["tuple_output"]()
tuple_trace = tl.trace(tuple_model, tuple_x, capture_container_structure=True)

tuple_op = tuple_trace["output_1"]
tuple_container = _cc.container_from_op(tuple_op)
print("tuple container kind:", tuple_container.kind)
print("len:", len(tuple_container))
print("summary:", tuple_container.summary())

## 7. `tl.register_container` — custom container types

TorchLens auto-detects `dict`, `tuple`, `list`, namedtuples, HF `ModelOutput`.  For any other
Python class you can register it with custom flatten/unflatten callables.

In [ ]:
import inspect

print("register_container signature:")
print(inspect.signature(tl.register_container))
print()
print("Parameters:")
print("  container_type : runtime class to recognize as a container")
print("  flatten        : callable returning (children, aux_data)")
print("  unflatten      : callable accepting (aux_data, children) -> instance")

## 8. `Trace.to_pandas()` and `Layer.to_pandas()` / `Op.to_pandas()`

Every record type exposes `to_pandas()` returning a single-row `DataFrame`.
`Trace.to_pandas()` returns all ops as a multi-row `DataFrame`.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

# Trace-level DataFrame
trace_df = trace.to_pandas()
print("trace.to_pandas() shape:", trace_df.shape)
print("columns (first 12):", list(trace_df.columns[:12]))
print()
print(trace_df[["layer_label", "layer_type", "activation_memory"]].to_string())

In [ ]:
# Layer-level DataFrame (86 columns — layer aggregated view)
layer = trace.layers["relu_1_2"]
layer_df = layer.to_pandas()
print("Layer.to_pandas() shape:", layer_df.shape)
print("columns (first 8):", list(layer_df.columns[:8]))
print()
print(layer_df[["layer_label", "layer_type", "num_passes"]].to_string())

In [ ]:
# Op-level DataFrame (181 columns — pass-level detail)
op = trace.ops["relu_1_2:1"]
op_df = op.to_pandas()
print("Op.to_pandas() shape:", op_df.shape)
print("columns (first 8):", list(op_df.columns[:8]))
print()
# Show a selection of meaningful columns
show_cols = [
    c
    for c in ["label", "layer_type", "activation_memory", "func", "grad_fn_name"]
    if c in op_df.columns
]
print(op_df[show_cols].to_string())

## 9. `Trace.output_table()` — semantic output decoding

`output_table` returns a `DataFrame` of decoded logits/predictions.  It requires the
trace to have been captured with semantic output decoding enabled and logits retained.
On a vanilla capture it raises a `ValueError`.

In [ ]:
import inspect

print("output_table signature:", inspect.signature(trace.output_table))
print()

try:
    tbl = trace.output_table()
    print(tbl)
except ValueError as exc:
    print("⚠️  output_table() on vanilla capture raises ValueError:")
    print(f"    {exc}")
    print()
    print("Requires: trace captured with semantic output decoding + retained logits.")

---

## ⚠️ GAPs / ergonomic smells

- **`Container` is not discoverable from a plain `tl.trace` call.** Users must know
  (a) to pass `capture_container_structure=True`, AND (b) to call `_cc.container_from_op(op)`
  from the semi-private `torchlens.data_classes.container` module.  There is no
  `trace.containers` property or `op.container` shortcut that works on a standard trace.
  The public `tl.Container` class is in `__all__` but no public factory creates instances.
  **Ergonomic smell:** `capture_container_structure=True` should probably default `True`,
  or `trace.containers` should be a top-level accessor.

- **`tl.Container.summary()` == `repr()`.** The method is present but its implementation
  just calls `repr(self)`, giving the same one-block string both ways.  Not a bug,
  but worth noting for JMT: summary / repr distinction is erased here.

- **`Trace.output_table()` errors on a vanilla trace** with a message about retained logits.
  The error is clear, but there is no easy code path in the docs/examples showing how to
  enable it; needs docs/example.

- **`Layer.to_pandas()` (86 cols) vs `Op.to_pandas()` (181 cols) vs `Trace.to_pandas()` (153 cols)
  have different column sets** — none is a superset of another.  A user who calls each hoping
  for the same schema will be confused.

- **`tl.extract` dict mapping direction is undocumented.** The convention is
  `{alias: real_label}` (alias is the output key, real_label is the TorchLens layer label).
  Passing `{real_label: alias}` silently errors with "Layer not found" — easy to get
  backwards without docs.

- No GAPs in `tl.extract` / `tl.batched_extract` core functionality — both run green.